# 응용 1. 직선 외삽으로 배 위치 밀기 (직선 DR)

**개념:** [개념 1 외삽 공식](../01_concepts/01_extrapolation.ipynb)

개념 1의 **선형 외삽**을 바다 위 배에 쓴다. 항해에서는 이것을 **추측항법(Dead Reckoning, DR)**이라고 부른다.

가정은 같다: *"마지막 보고의 속력과 방향으로 곧게 계속 간다."* 달라지는 점은 두 가지다.
- 값이 하나가 아니라 **동쪽·북쪽 두 방향(2차원)**이다.
- 기울기(속도)를 두 점으로 구하지 않고, AIS가 알려 주는 **속력(SOG)·방향(COG)**을 그대로 쓴다.

순서
1. 속력에 방향 붙이기
2. 방향을 북·동으로 나누기
3. 직선 DR 함수
4. m를 위도·경도로 바꾸기
5. 가짜 배로 시험하기

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sim

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
KN = sim.KN

## 1. 속력에 방향 붙이기

선형 외삽을 시간으로 쓰면 $x = x_0 + v\tau$다. 바다에서는 배가 평면 위를 움직이므로 **방향**이 하나 더 필요하다.

| AIS 값 | 뜻 | 단위 |
|---|---|---|
| **SOG** (Speed Over Ground) | 땅 기준 실제 속력 | 노트 |
| **COG** (Course Over Ground) | 땅 기준 실제 진행 방향. 북쪽 0°, 시계 방향 | 도 |

## 2. 방향을 북·동으로 나누기

속력 $v$로 방향 $C$를 향해 가면, 1초마다 북쪽과 동쪽으로 이만큼 간다.

$$v_n = v\cos C, \qquad v_e = v\sin C$$

수학의 각도(동쪽 0°, 반시계)와 달리 **북쪽 0°, 시계 방향**이라 `sin`과 `cos` 자리가 바뀐다.

| COG | 북쪽 $\cos C$ | 동쪽 $\sin C$ |
|---|---|---|
| 0° (북) | 1 | 0 |
| 90° (동) | 0 | 1 |
| 180° (남) | −1 | 0 |
| 45° (북동) | 0.707 | 0.707 |

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
for cog in [0, 45, 90, 135, 200, 300]:
    c = np.radians(cog)
    ax.arrow(0, 0, np.sin(c), np.cos(c), head_width=0.06, length_includes_head=True, color="C0")
    ax.text(1.15 * np.sin(c), 1.15 * np.cos(c), f"{cog}°", ha="center", va="center")
ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4); ax.set_aspect("equal")
ax.set_xlabel("동쪽"); ax.set_ylabel("북쪽"); ax.grid(alpha=.3)
ax.set_title("COG별 진행 방향 (sin C, cos C)"); plt.show()

## 3. 직선 DR 함수

$$e(\tau) = e_0 + v\sin C\cdot\tau, \qquad n(\tau) = n_0 + v\cos C\cdot\tau$$

$\tau$는 마지막 보고 이후 지난 시간(초), $v$는 SOG를 m/s로 바꾼 값이다. 선형 외삽을 방향별로 두 번 한 것이다.

In [ ]:
def dead_reckon(e0, n0, sog_kn, cog_deg, tau):
    """마지막 보고 (e0, n0, SOG, COG)에서 tau 초 동안 곧게 간 위치 (m)."""
    v = sog_kn * KN
    c = np.radians(cog_deg)
    return e0 + v * np.sin(c) * tau, n0 + v * np.cos(c) * tau

# 12노트, 북동(045°)으로 5분
e, n = dead_reckon(0, 0, 12, 45, 5 * 60)
print(f"동쪽 {e:.1f} m, 북쪽 {n:.1f} m, 거리 {np.hypot(e, n):.1f} m = {np.hypot(e, n) / 1852:.2f} 해리")

## 4. m를 위도·경도로 바꾸기

지도에 그리려면 "북쪽 1309m, 동쪽 1309m"를 위도·경도로 바꿔야 한다. 짧은 거리에서는 지구를 평평하게 봐도 된다.

$$\Delta\varphi = \frac{\Delta n}{R}, \qquad \Delta\lambda = \frac{\Delta e}{R\cos\varphi}$$

정확하게 하려면 marine-algorithms의 **Vincenty 정문제**(출발점 + 방위 + 거리 → 도착점)를 쓴다. 직선 DR은 결국 *"거리 = SOG × 시간"을 구해서 정문제에 넣는 것*이다.

In [ ]:
R = 6_371_008.8  # 지구 평균 반지름 (m)

def offset_to_latlon(lat0, lon0, de, dn):
    lat = lat0 + np.degrees(dn / R)
    lon = lon0 + np.degrees(de / (R * np.cos(np.radians(lat0))))
    return lat, lon

lat, lon = offset_to_latlon(35.1028, 129.0403, e, n)   # 부산항에서
print(f"도착: 위도 {lat:.5f}, 경도 {lon:.5f}")

## 5. 가짜 배로 시험하기

`sim.py`가 정답을 아는 가짜 배를 만든다.

- 12노트로 북쪽으로 가다가 **200~380초에 1°/초로 우회전**(180° 돌기), 그 뒤 다시 직진
- **20초마다** AIS 보고. 위치 ±5m, SOG ±0.2kn, COG ±2° 잡음

앱은 보고를 받을 때마다 그 값으로 다음 보고 전까지 배를 밀어 그린다. 이 *민 위치*가 실제와 몇 m 틀리는지 직진·선회 구간별로 잰다.

In [ ]:
rep = sim.make_reports()

def predict_dr(k, tau):
    """보고 k 의 원래 SOG·COG로 tau 초 곧게 민 위치."""
    v, c = rep["sog"][k] * KN, np.radians(rep["cog"][k])
    return rep["e"][k] + v * np.sin(c) * tau, rep["n"][k] + v * np.cos(c) * tau

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, (xl, yl), title in zip(axes, sim.ZOOM_VIEWS, ["직진 구간 (가로를 늘려 봄)", "선회 구간"]):
    sim.plot_truth(ax, rep)
    sim.plot_extrapolation(ax, rep, predict_dr, "직선 DR", "C3")
    ax.set_xlim(*xl); ax.set_ylim(*yl); ax.set_title(title); ax.legend(loc="lower right")
axes[0].set_aspect("auto"); axes[1].legend(loc="lower center")
plt.show()

print(sim.evaluate(rep, predict_dr))

## 두 가지 약점 (개념 2와 같다)

| 약점 | 왜 생기나 | 그림에서 | 보완 |
|---|---|---|---|
| **잡음** | 보고 원래 값을 그대로 믿는다. COG가 2° 틀리면 20초 뒤 약 4m 옆으로 샌다 | 직진인데도 선이 비뚤고, 새 보고에서 툭 끊긴다 | 응용 2 칼만 필터 |
| **선회** | "곧게 간다"는 가정이 틀렸다 | 선회 중 선이 원 바깥으로 뻗는다 | 응용 3 확장 칼만 필터 |

## 직접 해 보기

1. `sim.REPORT_INTERVAL = 60`으로 바꾸고 `rep = sim.make_reports()`부터 다시 돌리면 선회 오차는 어떻게 되나?
2. `sim.COG_NOISE = 0`이면 직진 오차가 얼마나 줄어드나? 남는 오차는 어디서 오나?

In [ ]:
# 여기서 직접 바꿔 보세요
